# Workshop Apache Spark and Batch Processing

## Outline

- [Introduction](#introduction)
- [Technologies](#technologies)
- [Apache Spark](#apache-spark)
- [Installation](#installation)
- [First Look](#first-look)
- [Spark Internals](#spark-internals)

<hr>

### Introduction

Generally one has two options to process data.

- Batch
- Streaming

**Batch**

When talking about batch processing, one usually refers to the process of fetching or loading data in discrete units. These units can take on many forms such as time periods for example. Advantages are that batch processed processes are easy to maintain and manage. Furthermore processes can be retried and is easier to scale. One disadvantage might be time delay

**Streaming**

Streaming processes the data in a more 'fluent' manner. The data is processed immidiately so that the data flow is not interrupted. A common example would be the streaming of videos or songs on YouTube or Spotify where the medium pre loads a certain amount of time. When this specific timepoint is reached, the loading process is continued.

### Technologies

Different technologies can be leveraged to perform batch processing. Examples are:

- Python Scripts
- SQL
- Spark
- Flink

The workflow is often orchestrated by the popular technology apache airflow

### Apache Spark

Apache Spark is a distributed data processing framework (multilingual engine) designed for fast, large-scale data analytics. It runs computations across a cluster of machines by splitting data into partitions and processing them in parallel.

Spark applications consist of a driver program, which coordinates execution, and executors, which perform tasks on worker nodes. Data is processed using high-level APIs (Scala, Python, Java, SQL) built around distributed collections called RDDs or optimized DataFrames/Datasets.

Operations are divided into:

Transformations (lazy operations that define computation steps)

Actions (operations that trigger execution and return results)

Spark builds a Directed Acyclic Graph (DAG) of transformations, optimizes it, and schedules tasks across the cluster. It keeps data in memory whenever possible, making it much faster than disk-based systems like traditional MapReduce.

### Installation

To install pyspark, we first need to get Java because Pyspark requires Java 17 or a later version. I ran the following command

Then I configured the downloaded java file and added it to the Path

After successfully downloading Java I created a virtual environment with uv and added pyspark to the available packages.

### First Look

For a first look I followed the videotutorial, accessible at [this page](https://www.youtube.com/watch?v=r_Sf6fCB40c&list=PL3MmuxUbc_hJed7dXYoJw8DoCuVHhGEQb&index=55)

First let's import the necessary libraries

In [ ]:
import pyspark
from pyspark.sql import SparkSession
import pandas as pd

In [3]:
spark = SparkSession.builder.master("local[*]").appName("test").getOrCreate()
        

Downloading some data (High volume taxi data from January 2021)

In [6]:
!curl -o fhvhv_tripdata_2021-01.parquet https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2021-01.parquet

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0  294M    0  371k    0     0   735k      0  0:06:50 --:--:--  0:06:50  743k
  2  294M    2 7537k    0     0  5005k      0  0:01:00  0:00:01  0:00:59 5021k
  5  294M    5 15.2M    0     0  6248k      0  0:00:48  0:00:02  0:00:46 6262k
  7  294M    7 22.0M    0     0  6422k      0  0:00:46  0:00:03  0:00:43 6431k
  9  294M    9 29.0M    0     0  6530k      0  0:00:46  0:00:04  0:00:42 6538k
 12  294M   12 36.4M    0     0  6786k      0  0:00:44  0:00:05  0:00:39 7398k
 14  294M   14 43.3M    0     0  6826k      0  0:00:44  0:00:06  0:00:38 7375k
 16  294M   16 49.9M    0     0  6802k      0  0:00:44  0:00:07  0:00:37 7079k
 19  294M   19 56.2M    0     0  6780k      0  0:00:44  0:00:08  0:00:36 7031k
 21  294M   21 62.4M    0     0  6724k      0  0:00

Checking out the dataframe:

In [6]:
df = spark.read.option("header", "true").parquet("fhvhv_tripdata_2021-01.parquet")

#print the dataframe
df.show()

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+--

In [9]:
df.head(5)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', originating_base_num='B02682', request_datetime=datetime.datetime(2021, 1, 1, 0, 28, 9), on_scene_datetime=datetime.datetime(2021, 1, 1, 0, 31, 42), pickup_datetime=datetime.datetime(2021, 1, 1, 0, 33, 44), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 49, 7), PULocationID=230, DOLocationID=166, trip_miles=5.26, trip_time=923, base_passenger_fare=22.28, tolls=0.0, bcf=0.67, sales_tax=1.98, congestion_surcharge=2.75, airport_fee=None, tips=0.0, driver_pay=14.99, shared_request_flag='N', shared_match_flag='N', access_a_ride_flag=' ', wav_request_flag='N', wav_match_flag='N'),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', originating_base_num='B02682', request_datetime=datetime.datetime(2021, 1, 1, 0, 45, 56), on_scene_datetime=datetime.datetime(2021, 1, 1, 0, 55, 19), pickup_datetime=datetime.datetime(2021, 1, 1, 0, 55, 19), dropoff_datetime=datetime.datetime(2021, 1, 1, 1, 18, 21), PULocationID=152, DO

Creating test file

In [ ]:
!head -n 101 fhvhv_tripdata_2021-01.parquet > head.parquet

In [ ]:
!wc -l head.parquet

In [13]:
df_pandas = pd.read_parquet("head.parquet")
df_pandas.dtypes()

ArrowKeyError: No type extension with name arrow.py_extension_type found

Creating Schema

In [ ]:
from pyspark.sql import types

In [ ]:
df.schema()

In [ ]:
!head -n 100 fhvhv_tripdata_2021-01.parquet > head.csv
!wc -l head.csv

In [ ]:
df_pandas = pd.read_csv("head.csv")

In [ ]:
df_pandas.dtypes

Turning this dataframe into a spark dataframe

In [ ]:
spark.createDataFrame(df_pandas).schema

## Spark Internals

Spark Internals refer to Spark Clusters. Spark Clusters are interconnected computers (nodes) that run Apache Spark to process massive datasets in parallel.

In the case above we have a local master computer (connected with Driver). The master has a control function and has an overview over what jobs are running. It can also send instructions on what jobs should be run.

With Spark Submit we can send a request for example what kind of resources we need.

The third part are the executors, machines that do the actual computation. For example, executors can pull data (partition) from a source and process that data. There is also the option to use Hadoop or HDFS. It is more memory efficient. These tools rather pull code and store data in the executors.

**Group By**

This command happens in two stages in spark internally. First the records are grouped by in every partition. In the second stage reshuffling is applied, that in each partition only records with the same key are present.

**Reshuffling**

Process that shuffles the records we have in each partition, moving records between the various partitions. The algorithm applied is called External Merge Sort. 

Shuffling is done when some data transformation operation occur, like groupby or join.